# 3. Chunking y Embeddings con LangChain y GitHub Models

## Objetivos de Aprendizaje
- Entender la necesidad de dividir documentos largos en fragmentos (chunking).
- Aprender a usar los divisores de texto de LangChain (`RecursiveCharacterTextSplitter`).
- Comprender qué son los embeddings y por qué son fundamentales para la búsqueda semántica.
- Generar embeddings para los chunks de texto utilizando los modelos de GitHub a través de la API de OpenAI.

## De la Búsqueda por Palabras Clave a la Búsqueda Semántica

En el notebook anterior, construimos un RAG básico que funcionaba con palabras clave. Sus limitaciones eran claras: no entendía el significado, los sinónimos o el contexto. Para superar esto, necesitamos dos procesos clave:

1.  **Chunking**: Si un documento es muy largo, su embedding representará una idea demasiado general. Al dividirlo en chunks más pequeños y cohesivos, cada chunk representa una idea más específica. Esto hace que la búsqueda de similitud sea mucho más precisa.
2.  **Embeddings**: Son la pieza central de la búsqueda semántica. Convierten el texto (nuestros chunks) en vectores numéricos en un espacio multidimensional. Los textos con significados similares tendrán vectores cercanos en este espacio, lo que nos permite encontrar información relevante aunque no se usen las mismas palabras.

Si tienen problemas con las dependencias de las librerias, se recomienda crear un nuevo ambiente de python usnado el siguiente codigo:

conda create -n rag_env python=3.11
conda activate rag_env
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install langchain langchain-text-splitters langchain-groq groq openai jupyter
jupyter notebook

In [1]:
# Celda 2: Importaciones y configuración
import os
from openai import OpenAI
import json

# Importar las bibliotecas necesarias
from groq import Groq
from dotenv import load_dotenv
from openai import OpenAI
import os

# load .env
load_dotenv()


# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)


print("✅ Librerías importadas correctamente")

OpenAI library version: 2.29.0
Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
✅ Librerías importadas correctamente


In [2]:
import os
import string
from openai import OpenAI
#from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, Markdown
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# --- Configuración del Cliente de OpenAI ---
def initialize_client():
    # Initialize Groq LLM
    llm = ChatGroq(
        api_key=os.getenv("GROQ_API_KEY"),
        model_name="llama-3.1-8b-instant",
        temperature=0.7
    )
    return llm

client = initialize_client()
print("✓ Cliente de OpenAI (compatible con GitHub Models) inicializado.")

c:\Users\Lenovo\anaconda3\envs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Cliente de OpenAI (compatible con GitHub Models) inicializado.


## 1. El Documento a Procesar

Comenzamos con un documento de texto más largo que los que usamos en el notebook anterior. Este texto sobre la historia de la inteligencia artificial es ideal para demostrar la necesidad del chunking.

In [3]:
long_text = (
    "La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. "
    "Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. "
    "El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. "
    "Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. "
    "Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. "
    "El resurgimiento llegó en la década de 1980 con el auge de los sistemas expertos, programas que encapsulaban el conocimiento de un experto humano en un dominio específico, como el diagnóstico médico (por ejemplo, MYCIN). "
    "Estos sistemas demostraron el valor comercial de la IA, pero su fragilidad y el alto costo de mantenimiento condujeron a un segundo invierno a finales de los 80 y principios de los 90. "
    "La revolución moderna de la IA comenzó a gestarse a finales de los 90 y principios de los 2000, impulsada por tres factores clave: la disponibilidad de grandes volúmenes de datos (Big Data), el desarrollo de hardware más potente (especialmente las GPU) y los avances en algoritmos de aprendizaje automático, en particular las redes neuronales profundas (deep learning). "
    "Hitos como la victoria de Deep Blue de IBM sobre el campeón de ajedrez Garry Kasparov en 1997 y, más tarde, el triunfo de AlphaGo de DeepMind en el juego de Go en 2016, demostraron el poder del aprendizaje por refuerzo y el deep learning. "
    "Hoy, vivimos en la era de los modelos de lenguaje grande (LLM) como GPT y Claude, y los modelos de difusión para la generación de imágenes, que han llevado la IA a la corriente principal, transformando industrias y planteando nuevas preguntas sobre el futuro de la tecnología y la humanidad."
)

display(Markdown("### 📜 Documento Original"))
print(long_text)
print(f"Longitud del texto: {len(long_text)} caracteres.")

### 📜 Documento Original

La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. El resurgimiento llegó en la década de 1980 con el auge de los sistemas expertos, programas que encapsulaban el conocimiento de un experto humano en un domini

## 2. Chunking con `RecursiveCharacterTextSplitter`

LangChain ofrece varias herramientas para dividir texto. `RecursiveCharacterTextSplitter` es una de las más recomendadas porque intenta dividir el texto basándose en una jerarquía de separadores (como saltos de línea dobles, saltos de línea simples, espacios, etc.) para mantener los fragmentos lo más coherentes posible.

-   `chunk_size`: Define el tamaño máximo de cada chunk (en caracteres).
-   `chunk_overlap`: Define cuántos caracteres se superponen entre chunks consecutivos. Esto es crucial para no perder el contexto que podría existir justo en el límite entre dos chunks.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Inicializar el divisor de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,  # Tamaño del chunk en caracteres
    chunk_overlap=50,  # Superposición para mantener contexto
    length_function=len
)

# 2. Dividir el documento en chunks
chunks = text_splitter.split_text(long_text)

display(Markdown(f"### 🧩 Documento Dividido en {len(chunks)} Chunks"))

# 3. Mostrar los chunks resultantes
for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i+1} (Longitud: {len(chunk)}) ---")
    print(chunk)
    print()

### 🧩 Documento Dividido en 7 Chunks

--- CHUNK 1 (Longitud: 349) ---
La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia

--- CHUNK 2 (Longitud: 349) ---
John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. Sin embargo, las

--- CHUNK 3 (Longitud: 348) ---
lógica y teoremas matemáticos. Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de r

In [5]:
chunks

["La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia",
 'John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. Sin embargo, las',
 "lógica y teoremas matemáticos. Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. El resurgimiento llegó en la década de 1980 con el au

## 3. Generación de Embeddings para cada Chunk

Ahora que tenemos nuestros chunks, el siguiente paso es convertirlos en vectores numéricos. Usaremos el modelo `text-embedding-3-small`, que es eficiente y potente.

El resultado será una lista de vectores, donde cada vector corresponde a un chunk de nuestro documento.

**Importante**: Install the pip install sentence-transformers 

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embeddings(texts):
    if isinstance(texts, str):
        texts = [texts]
    embeddings = model.encode(texts)  # ← sin convert_to_list
    return embeddings.tolist()        # ← convertir numpy array a lista


# Generar embeddings para todos los chunks
try:
    chunk_embeddings = get_embeddings(chunks)
    print(f"✓ Se generaron {len(chunk_embeddings)} embeddings.")
    print(f"Dimensión de cada embedding: {len(chunk_embeddings[0])}")
except Exception as e:
    print(f"❌ Error al generar embeddings: {e}")

# Mostrar ejemplo
if 'chunk_embeddings' in locals() and chunk_embeddings:
    display(Markdown("### 🔍 Ejemplo: Chunk y su Embedding"))
    
    example_index = 0
    example_chunk = chunks[example_index]
    example_embedding = chunk_embeddings[example_index]
    
    print(f"**Texto del Chunk {example_index + 1}:** {example_chunk}")
    print(f"**Embedding (primeros 10 de {len(example_embedding)} dimensiones):** {example_embedding[:10]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7760.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Se generaron 7 embeddings.
Dimensión de cada embedding: 384


### 🔍 Ejemplo: Chunk y su Embedding

**Texto del Chunk 1:** La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia
**Embedding (primeros 10 de 384 dimensiones):** [-0.035220373421907425, 0.015380745753645897, -0.049861546605825424, -0.02010507695376873, -0.050072647631168365, -0.02158275619149208, -0.0030390529427677393, 0.06407622992992401, -0.02647963911294937, 0.10094454884529114]...


In [11]:
chunk_embeddings[0]

[-0.035220373421907425,
 0.015380745753645897,
 -0.049861546605825424,
 -0.02010507695376873,
 -0.050072647631168365,
 -0.02158275619149208,
 -0.0030390529427677393,
 0.06407622992992401,
 -0.02647963911294937,
 0.10094454884529114,
 0.042960863560438156,
 -0.002676671138033271,
 0.05804094299674034,
 0.007653126027435064,
 -0.1059194803237915,
 0.04617994651198387,
 -0.07607053220272064,
 -0.043074220418930054,
 -0.025785036385059357,
 -0.03174814209342003,
 0.04311071336269379,
 -0.02290748991072178,
 -0.019942915067076683,
 0.023050915449857712,
 -0.01878487877547741,
 0.10727221518754959,
 -0.04347046837210655,
 -0.07713243365287781,
 -0.027843305841088295,
 -0.003974331542849541,
 -0.053943827748298645,
 0.03437182307243347,
 0.1661999523639679,
 -0.07451698929071426,
 0.03618909791111946,
 0.01180208195000887,
 0.03506939485669136,
 0.047539133578538895,
 0.09703495353460312,
 -0.016408225521445274,
 -0.04422513395547867,
 -0.10541713237762451,
 0.026960093528032303,
 0.004916986

## 4. Conclusiones y Próximos Pasos

¡Felicidades! Has completado los dos pasos fundamentales para pasar de un RAG básico a uno semántico:

1.  **Has dividido un documento largo en chunks manejables y coherentes.**
2.  **Has convertido cada chunk de texto en un vector numérico (embedding) que captura su significado.**

Con esta lista de chunks y su correspondiente lista de embeddings, ahora tienes una **base de conocimiento vectorial**. 

**En el próximo notebook (`3-vector-rag.ipynb`), aprenderás a:**
-   Tomar una consulta del usuario y generar también su embedding.
-   Usar la **similitud coseno** para comparar el vector de la consulta con todos los vectores de los chunks.
-   Seleccionar los chunks más similares (los que tienen la puntuación de similitud más alta) para usarlos como contexto en la generación de la respuesta.

In [17]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calcula la similitud coseno entre dos vectores."""
    vec1, vec2 = np.array(vec1), np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def find_most_similar_chunks(query, chunks, chunk_embeddings, top_k=3):
    """Busca los chunks más similares a la query."""
    
    # 1. Convertir la query a embedding
    query_embedding = get_embeddings(query)[0]
    
    # 2. Calcular similitud coseno con cada chunk
    similarities = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_emb)
        similarities.append((i, score, chunks[i]))
    
    # 3. Ordenar por similitud (mayor a menor)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]


# ─── PRUEBA ───────────────────────────────────────────
query = "historia sobre la AI" 

resultados = find_most_similar_chunks(query, chunks, chunk_embeddings, top_k=1)

display(Markdown(f"### 🔍 Query: *{query}*"))
print(f"\nTop {len(resultados)} chunks más relevantes:\n")

for rank, (idx, score, texto) in enumerate(resultados, 1):
    print(f"--- #{rank} | Chunk {idx+1} | Similitud: {score:.4f} ---")
    print(texto)
    print()

### 🔍 Query: *historia sobre la AI*


Top 1 chunks más relevantes:

--- #1 | Chunk 1 | Similitud: 0.5420 ---
La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia



In [16]:
chunks

["La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia",
 'John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. Sin embargo, las',
 "lógica y teoremas matemáticos. Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. El resurgimiento llegó en la década de 1980 con el au